<a href="https://colab.research.google.com/github/useDeep/learn_ml/blob/main/learn_hf_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import pipeline

pipe = pipeline("text-generation", model="openai-community/gpt2")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [2]:
output= pipe("THe cat sat", max_length=40)
output

[{'generated_text': 'THe cat sat down next to him, and suddenly a large, large, black cat appeared in front of his eyes. He looked like a man out of a movie, with a big, purple'}]

In [3]:
print(output[0]["generated_text"])

THe cat sat down next to him, and suddenly a large, large, black cat appeared in front of his eyes. He looked like a man out of a movie, with a big, purple


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", device_map="auto")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [5]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [6]:
tokenizer

GPT2Tokenizer(name_or_path='openai-community/gpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [7]:
text= "This is a great"
input_ids= tokenizer(text, return_tensors= "pt")["input_ids"]
input_ids

tensor([[1212,  318,  257, 1049]])

In [8]:
tokenizer.decode(input_ids)

['This is a great']

In [9]:
for token_id in input_ids[0]:
  print(f"\"{tokenizer.decode(token_id)}\" - {token_id}")

"This" - 1212
" is" - 318
" a" - 257
" great" - 1049


In [10]:
tokenizer("My")["input_ids"]

[3666]

In [11]:
tokenizer(" My")["input_ids"]

[2011]

In [12]:
tokenizer("My ")["input_ids"]

[3666, 220]

In [13]:
tokenizer("mY")["input_ids"]

[76, 56]

In [14]:
tel= tokenizer("అ")["input_ids"]
tel

[156, 108, 227]

In [15]:
for token_id in tel:
  print(f"\"{tokenizer.decode(token_id)}\" - {token_id}")

"�" - 156
"�" - 108
"�" - 227


In [16]:
input= tokenizer("The cat sat", return_tensors= "pt")["input_ids"].to(model.device)
print(input)

output= model.generate(input, max_length= 50)
print(output)

tensor([[ 464, 3797, 3332]])
tensor([[  464,  3797,  3332,   319,   262,  4314,    11,   290,   262,  3797,
           373,   991, 16039,    13,   198,   198,     1,    40,  1101,  7926,
            11,   314,  1101,  7926,   553,   673,   531,    13,   366,    40,
          1101,  7926,    13,   314,  1101,  7926,    13,   314,  1101,  7926,
            13,   314,  1101,  7926,    13,   314,  1101,  7926,    13,   314]])


In [48]:
input= tokenizer("The cat sat", return_tensors= "pt").to(model.device)
print(input)

output= model.generate(**input, max_length= 50)
print(output)

{'input_ids': tensor([[ 464, 3797, 3332]]), 'attention_mask': tensor([[1, 1, 1]])}
tensor([[  464,  3797,  3332,   319,   262,  4314,    11,   290,   262,  3797,
           373,   991, 16039,    13,   198,   198,     1,    40,  1101,  7926,
            11,   314,  1101,  7926,   553,   673,   531,    13,   366,    40,
          1101,  7926,    13,   314,  1101,  7926,    13,   314,  1101,  7926,
            13,   314,  1101,  7926,    13,   314,  1101,  7926,    13,   314]])


In [53]:
out= model(input.input_ids)   # for logits
out

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[ -36.2874,  -35.0113,  -38.0793,  ...,  -40.5163,  -41.3759,
           -34.9193],
         [ -71.8221,  -72.3168,  -74.6482,  ...,  -80.4650,  -74.6845,
           -74.2299],
         [-104.3910, -105.4351, -112.6575,  ..., -115.7211, -107.2959,
          -108.3043]]], grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

In [56]:
out= out.logits
out

tensor([[[ -36.2874,  -35.0113,  -38.0793,  ...,  -40.5163,  -41.3759,
           -34.9193],
         [ -71.8221,  -72.3168,  -74.6482,  ...,  -80.4650,  -74.6845,
           -74.2299],
         [-104.3910, -105.4351, -112.6575,  ..., -115.7211, -107.2959,
          -108.3043]]], grad_fn=<UnsafeViewBackward0>)

In [57]:
out= torch.softmax(out[0, -1], dim= -1)
out

tensor([4.0192e-05, 1.4148e-05, 1.0328e-08,  ..., 4.8255e-10, 2.2007e-06,
        8.0282e-07], grad_fn=<SoftmaxBackward0>)

In [18]:
result= tokenizer.decode(output[0])
print(result)

The cat sat on the floor, and the cat was still asleep.

"I'm sorry, I'm sorry," she said. "I'm sorry. I'm sorry. I'm sorry. I'm sorry. I'm sorry. I


In [19]:
print(text)
input_ids

This is a great


tensor([[1212,  318,  257, 1049]])

In [20]:
model(input_ids).logits[0, -1]    # last logit or first prediction

tensor([-105.2774, -105.3633, -110.4340,  ..., -113.0601, -114.6163,
        -108.1165], grad_fn=<SelectBackward0>)

In [21]:
output= model(input_ids).logits[0, -1]
tokenizer.decode(output.argmax())

' way'

In [22]:
out= model.generate(input_ids)

tokenizer.decode(out)

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=24) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


['This is a great way to get a little bit of a feel for the game.\n\nThe game is a bit']

In [23]:
import torch

out= model(input_ids).logits[0, -1]
final_logits= torch.topk(out, 10)
final_logits

torch.return_types.topk(
values=tensor([-97.2303, -97.6799, -97.8907, -97.9865, -98.5808, -98.6885, -98.8354,
        -98.8833, -99.0749, -99.1456], grad_fn=<TopkBackward0>),
indices=tensor([ 835, 1672, 3663, 2126, 1295,  640, 1730, 2891,  923, 1621]))

In [24]:
# text= "This is a great"
for i in final_logits.indices:
  print(tokenizer.decode(i))

 way
 example
 opportunity
 idea
 place
 time
 deal
 tool
 start
 story


In [25]:
out= model(input_ids).logits[0, -1]
len(out)

50257

In [26]:
import torch.nn.functional as F

# Sampling methods

def greedy_decode(logits):
  return torch.argmax(logits, dim= -1)

def top_k(logits, k= 10):
  logits, indices= torch.topk(logits, k)
  print(logits)
  probs= F.softmax(logits, dim= -1)
  sampled= torch.multinomial(probs, 1)
  return indices[sampled]

def top_p(logits, p= 0.9):
  sorted_logits, sorted_indices= torch.sort(logits, descending= True)
  sorted_probs= F.softmax(sorted_logits, dim= -1)
  cumulative_probs= sorted_probs.cumsum(dim= -1)

  # Mask probs outside nuclues
  mask= cumulative_probs > p
  sorted_logits[mask]= float("-inf")

  filtered_probs= F.softmax(sorted_logits, dim= -1)
  sampled= torch.multinomial(filtered_probs, 1)

  return sorted_indices[sampled]

def temperature(logits, temp= 1.0):
  scaled= logits/ temp
  probs= F.softmax(scaled, dim= -1)
  return torch.multinomial(probs, 1)

In [27]:
# text= "This is a great"

tokenizer.decode(greedy_decode(out))

' way'

In [28]:
tokenizer.decode(top_k(out, 5))

tensor([-97.2303, -97.6799, -97.8907, -97.9865, -98.5808],
       grad_fn=<TopkBackward0>)


' opportunity'

In [29]:
tokenizer.decode(top_p(out))

' issue'

In [30]:
tokenizer.decode(temperature(out, 8))

' height'

In [46]:
sentence= "The weather today"
token_ids= tokenizer(sentence, return_tensors= "pt").input_ids
outputs= model(token_ids).logits
outputs= torch.softmax(outputs[0, -1], dim= -1)

top10= torch.topk(outputs, k= 10)

for index, value in zip(top10.indices, top10.values):
  print(f"{tokenizer.decode(index)} --- {value:.1%}")

 is --- 25.3%
 was --- 21.7%
, --- 4.0%
 has --- 3.3%
 will --- 3.2%
 in --- 2.1%
. --- 1.6%
 and --- 1.4%
 could --- 1.2%
 may --- 1.1%


### Sentiment analysis

In [31]:
from datasets import load_dataset

ds= load_dataset("stanfordnlp/imdb")
ds

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [58]:
type(ds)

datasets.dataset_dict.DatasetDict

In [59]:
ds["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [60]:
import pandas as pd
ds["train"].to_pandas()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [61]:
pd= ds["train"].to_pandas()

In [62]:
pipe= pipeline("sentiment-analysis")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [63]:
pipe("A sad boy on a beautiful day")

[{'label': 'POSITIVE', 'score': 0.9996111989021301}]

In [69]:
pipe("love hate relationshp")

[{'label': 'NEGATIVE', 'score': 0.8671965599060059}]

In [70]:
pipe("మీరు భోజనం చేశారా?")

[{'label': 'NEGATIVE', 'score': 0.9667778015136719}]